# QIntent SDK - VS Code / Jupyter Test

Este notebook prueba el SDK público `qdsv-qintent` contra la API pública de QIntent/QDSV.

La idea es verificar tres cosas:

1. El paquete instala correctamente.
2. La API pública responde.
3. QIntent puede compilar y ejecutar consultas simples desde Jupyter.

In [ ]:
%pip install --upgrade --no-cache-dir qdsv-qintent==0.1.2

In [ ]:
from qintent import QIntentClient

client = QIntentClient()
spec = client.spec()

spec["status"], spec["product"], spec["package"]

## Ejemplo 1: seleccionar filas por umbral

In [ ]:
rows = [
    {"candidate_index": 0, "score": 720},
    {"candidate_index": 1, "score": 910},
    {"candidate_index": 2, "score": 840},
]

result = client.run(
    'find_rows("candidate_index").where("score", ">=", 850)',
    rows=rows,
)

result["status"], result["result"]["selected_rows"]

## Ejemplo 2: ranking y top_k

In [ ]:
rows = [
    {"candidate_index": 0, "quality_score": 720, "risk_ok": True},
    {"candidate_index": 1, "quality_score": 910, "risk_ok": True},
    {"candidate_index": 2, "quality_score": 860, "risk_ok": False},
    {"candidate_index": 3, "quality_score": 970, "risk_ok": True},
]

source = 'find_rows("candidate_index").where("quality_score", ">=", 850).rank_by("quality_score").top_k(2)'
result = client.run(source, rows=rows)

result["status"], result["result"]["selected_rows"]

## Ejemplo 3: QIntent Python-like sobre un dominio

In [ ]:
source = """
x = domain(0, 15)
score = max(x, 0)
find(x).where(all([x in [3, 7, 9], not x == 3])).rank_by(score).top_k(3)
"""

compiled = client.compile(source)
compiled["compiled_summary"]

In [ ]:
result = client.run(source)
result["status"], result["result"].get("ranked_candidates", result["result"])

## Docker/local privado

Si quieres apuntar al despliegue local Docker en vez de la API pública:

```python
local_client = QIntentClient.local()
local_client.spec()
```

Por defecto usa `http://localhost:18080/api`.